# NLP Practical Notebook: Offline and Minimal

**Dataset:** `data1.csv` with a `text` column. Run cells from top to bottom.

- No `try/except` blocks and no package/model downloads in the notebook.
- Unit IV uses a small rule-based coreference demonstration plus BLEU because a translation model is **not** listed as cached.
- BERT loads `google-bert/bert-base-uncased` from the local cache only.
- POS tagging, dependency parsing, NER, and WordNet require their NLTK data / spaCy English model to already be installed locally.

## 1. Imports and dataset

In [ ]:
import pandas as pd
import re
import spacy
import torch
from collections import *
from nltk import *
from nltk.tokenize import *
from nltk.stem import *
from nltk.corpus import wordnet as wn
from nltk.translate.bleu_score import *
from sklearn.feature_extraction.text import *
from sklearn.metrics.pairwise import *
from sklearn.preprocessing import *
from sklearn.neural_network import *
from transformers import *

df = pd.read_csv("data1.csv")
texts = df["text"].dropna().astype(str).tolist()
text = texts[0]
tokens = re.findall(r"[a-zA-Z]+", text.lower())
words = re.findall(r"[a-zA-Z]+", " ".join(texts).lower())
print("Dataset text:", text)

## UNIT I: Text preprocessing

In [ ]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
print("Tokens:", tokens)
print("Stems:", [stemmer.stem(w) for w in tokens])
print("Lemmas:", [lemmatizer.lemmatize(w) for w in tokens])

## UNIT I: N-gram language model and Laplace smoothing

In [ ]:
unigrams = Counter(words)
bigrams = Counter(zip(words[:-1], words[1:]))
w1, w2 = words[0], words[1]
V = len(unigrams)
print("Bigram probability:", bigrams[(w1, w2)] / unigrams[w1])
print("Laplace smoothing:", (bigrams[(w1, w2)] + 1) / (unigrams[w1] + V))

## UNIT I: BPE (one basic merge)

In [ ]:
pairs = Counter()
for word in words:
    pairs.update(zip(word, word[1:]))
pair = pairs.most_common(1)[0][0]
print("Most frequent character pair:", pair)
print("Merged token:", "".join(pair))

## UNIT I: Simple neural language model

In [ ]:
pairs = list(zip(words[:-1], words[1:]))
X_words = [a for a, b in pairs]
y_words = [b for a, b in pairs]
enc_x, enc_y = LabelEncoder(), LabelEncoder()
X = enc_x.fit_transform(X_words).reshape(-1, 1)
y = enc_y.fit_transform(y_words)
model = MLPClassifier(hidden_layer_sizes=(8,), max_iter=200, random_state=1)
model.fit(X, y)
last_word = words[-2]
pred = model.predict(enc_x.transform([last_word]).reshape(-1, 1))[0]
print("Next word:", enc_y.inverse_transform([pred])[0])

## UNIT II: POS tagging

In [ ]:
print(pos_tag(tokens))

## UNIT II: CFG and parsing

In [ ]:
sentence = "the boy saw the man".split()
grammar = CFG.fromstring("""
S -> NP VP
NP -> Det N
VP -> V NP
Det -> 'the'
N -> 'boy' | 'man'
V -> 'saw'
""")
for tree in ChartParser(grammar).parse(sentence):
    print(tree)

## UNIT II: PCFG and probabilistic parsing

In [ ]:
sentence = "the boy saw the man".split()
grammar = PCFG.fromstring("""
S -> NP VP [1.0]
NP -> Det N [1.0]
VP -> V NP [1.0]
Det -> 'the' [1.0]
N -> 'boy' [0.5] | 'man' [0.5]
V -> 'saw' [1.0]
""")
for tree in ViterbiParser(grammar).parse(sentence):
    print(tree)
    print("Probability:", tree.prob())

## UNIT II: Dependency parsing

In [ ]:
nlp = spacy.load("en_core_web_sm")
for token in nlp(text):
    print(token.text, "--", token.dep_, "--> ", token.head.text)

## UNIT III (1/3): WordNet

In [ ]:
word = next(w for w in tokens if wn.synsets(w))
synset = wn.synsets(word)[0]
print("Word:", word)
print("Meaning:", synset.definition())
print("Synonyms:", synset.lemma_names()[:5])

## UNIT III (2/3): Vector space model (TF-IDF)

In [ ]:
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(texts)
print("TF-IDF matrix shape:", X_tfidf.shape)
print("Feature words:", vectorizer.get_feature_names_out()[:10])

## UNIT III (3/3): Cosine similarity

In [ ]:
similarity = cosine_similarity(X_tfidf)
print("Cosine similarity matrix:")
print(similarity)

## UNIT IV (1/2): Basic coreference resolution

In [ ]:
nlp = spacy.load("en_core_web_sm")

coref_text = """
Ravi is a software engineer who works at a technology company.
He develops applications with his colleague Priya.
She is an expert in machine learning and often helps him solve difficult problems.
One day, Ravi and Priya discovered a bug in their project.
They investigated the issue together because it was affecting the application's performance.
After several hours, Ravi found the cause of the problem.
He explained his findings to Priya, and she suggested a better solution.
Both of them implemented the changes and tested the application.
The application worked perfectly, and their manager congratulated them.
He praised Ravi for identifying the bug and appreciated Priya for improving the solution.
The team celebrated their success because they had completed the project before the deadline.
"""

doc = nlp(coref_text)
names = list(dict.fromkeys(
    [ent.text for ent in doc.ents if ent.label_ == "PERSON"] +
    [token.text for token in doc if token.pos_ == "PROPN"]
))

for token in doc:
    if token.lower_ in ["he", "his", "him"]:
        print(token.text, "refers to", names[0] if names else None)
    elif token.lower_ in ["she", "her"]:
        print(token.text, "refers to", names[-1] if names else None)

## UNIT IV (2/2): BLEU evaluation metric

In [ ]:
reference = "the boy saw the man".split()
candidate = "the boy saw man".split()
score = sentence_bleu([reference], candidate,
                      smoothing_function=SmoothingFunction().method1)
print("BLEU score:", round(score, 4))

## UNIT V (1/2): Named Entity Recognition (NER)

In [ ]:
for entity in nlp(text).ents:
    print(entity.text, "->", entity.label_)

## UNIT V (2/2): BERT / Transformer

In [ ]:
model_name = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
model = AutoModel.from_pretrained(model_name, local_files_only=True)
inputs = tokenizer(text[:200], return_tensors="pt", truncation=True, padding=True)
with torch.no_grad():
    output = model(**inputs)
print("BERT output shape:", output.last_hidden_state.shape)

## Syllabus checklist

- **Unit I:** preprocessing, N-gram model, Laplace smoothing, BPE, neural language model.
- **Unit II:** POS tagging, CFG, PCFG, parsing, dependency parsing.
- **Unit III (3):** WordNet, TF-IDF vector space model, cosine similarity.
- **Unit IV (2):** basic rule-based coreference demonstration, BLEU evaluation.
- **Unit V (2):** NER, BERT / Transformer embeddings.

**Offline reminder:** This notebook does not download anything. `google-bert/bert-base-uncased` is loaded from cache only. The translation model `Helsinki-NLP/opus-mt-en-fr` was not listed as cached, so machine translation was replaced with coreference for the offline version. The spaCy model `en_core_web_sm` and NLTK resources (tagger and WordNet) must also already exist locally.